In [ ]:
import numpy as np
import pandas as pd
import os
import librosa
import matplotlib.pyplot as plt
import IPython
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation,Reshape,MaxPooling2D, Dropout, Conv2D, MaxPool2D, Flatten
from tensorflow.keras.utils import to_categorical

In [ ]:
import os

paths = []
labels = []

# Updated local directories
real_root_dir = r'C:\Users\CRESCENT\Desktop\Audio-DeepFake-Detection\input\Real audio'
fake_root_dir = r'C:\Users\CRESCENT\Desktop\Audio-DeepFake-Detection\input\Fake audio'

print('Paths defined')

In [ ]:
for filename in os.listdir(real_root_dir):
    file_path = os.path.join(real_root_dir, filename)
    paths.append(file_path)
    labels.append('real')
print('Real audio paths loaded')

In [ ]:
for filename in os.listdir(fake_root_dir):
    file_path = os.path.join(fake_root_dir, filename)
    paths.append(file_path)
    labels.append('fake')
print('Fake audio paths loaded')

In [ ]:
df = pd.DataFrame()
df['speech'] = paths
df['label'] = labels
print('Dataset loading complete')

In [ ]:
# Assuming specific files exist for testing; update names if needed
real_audio = os.path.join(real_root_dir, os.listdir(real_root_dir)[0])
fake_audio = os.path.join(fake_root_dir, os.listdir(fake_root_dir)[0])

In [ ]:
def extract_features(fake_root_dir, real_root_dir, max_length=500):
    features = []
    labels = []

In [ ]:
for file in os.listdir(fake_root_dir):
        file_path = os.path.join(fake_root_dir, file)
        try:
            audio, _ = librosa.load(file_path, sr=16000)
            mfccs = librosa.feature.mfcc(y=audio, sr=16000, n_mfcc=40)
            if mfccs.shape[1] < max_length:
                mfccs = np.pad(mfccs, ((0, 0), (0, max_length - mfccs.shape[1])), mode='constant')
            else:
                mfccs = mfccs[:, :max_length]
            features.append(mfccs)
            labels.append(1)  # 1 for fake
        except Exception as e:
            print(f"Error: {file_path}")
            continue

In [ ]:
for file in os.listdir(real_root_dir):
        file_path = os.path.join(real_root_dir, file)
        try:
            audio, _ = librosa.load(file_path, sr=16000)
            mfccs = librosa.feature.mfcc(y=audio, sr=16000, n_mfcc=40)
            if mfccs.shape[1] < max_length:
                mfccs = np.pad(mfccs, ((0, 0), (0, max_length - mfccs.shape[1])), mode='constant')
            else:
                mfccs = mfccs[:, :max_length]
            features.append(mfccs)
            labels.append(0)  # 0 for real
        except Exception as e:
            print(f"Error: {file_path}")
            continue
    return np.array(features), np.array(labels)

In [ ]:
x, y = extract_features(fake_root_dir, real_root_dir)
print("Features shape:", x.shape)
print("Labels shape:", y.shape)

In [ ]:
xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.2)

In [ ]:
# Updated for local context
directory = './models/' # Adjust if you have a local model folder
if os.path.exists(directory):
    for root, dirs, files in os.walk(directory):
        print(root)
        for file in files:
            print(f'File: {file}')

In [ ]:
# Model Creation Function - Part 1 (CNN Layers)
import tensorflow as tf
from tensorflow.keras import layers, Sequential, regularizers

def create_model(input_shape=(40, 500, 1)):
    model = Sequential([
        layers.InputLayer(input_shape=input_shape),
        layers.Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(2, 2)),

        layers.Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(2, 2)),

In [ ]:
# Model Creation Function - Part 2 (BiLSTM Layers)
layers.Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(2, 2)),

        layers.Reshape((-1, 128)),
        layers.Bidirectional(layers.LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.2)),
        layers.BatchNormalization(),
        layers.Bidirectional(layers.LSTM(128, dropout=0.3, recurrent_dropout=0.2)),

In [ ]:
# Model Creation Function - Part 3 (Dense & Compile)
layers.BatchNormalization(),
        layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
model = create_model()

In [ ]:
# Update path to where your .h5 file is stored locally
model_path = r'C:\Users\CRESCENT\Desktop\Audio-DeepFake-Detection\updated_model.h5'
if os.path.exists(model_path):
    model.load_weights(model_path)
    print("Model loaded successfully")

In [ ]:
from sklearn.metrics import precision_score, recall_score, confusion_matrix, accuracy_score

y_pred = model.predict(xtest)
y_pred_binary = (y_pred > 0.5).astype(int)

In [ ]:
precision = precision_score(ytest, y_pred_binary)
recall = recall_score(ytest, y_pred_binary)
accuracy = accuracy_score(ytest, y_pred_binary)

In [ ]:
print(f"Accuracy: {accuracy:.4f}")

In [ ]:
print(f"Precision: {precision:.4f}")

In [ ]:
print(f"Recall: {recall:.4f}")

In [ ]:
cm = confusion_matrix(ytest, y_pred_binary)
print("Confusion Matrix:")
print(cm)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negative', 'Positive'], 
            yticklabels=['Negative', 'Positive'])

In [ ]:
plt.ylabel('True labels')
plt.xlabel('Predicted labels')

In [ ]:
plt.title('Confusion Matrix')

In [ ]:
plt.show()

In [ ]:
print("Evaluation Complete.")